# OCR Quality Analysis

This notebook evaluates the OCR pipeline on cosmetic ingredient‑label images. It uses the existing `preprocess_image` and `extract_text` functions, compares OCR output against manually verified ground‑truth text, and reports Character Error Rate (CER) and Word Error Rate (WER).

The notebook is safe to run in Google Colab – it installs required system packages, checks for Tesseract availability, and works with an optional small demo dataset.

In [1]:
# Install system dependencies (Colab)
!apt-get update -qq && apt-get install -y -qq tesseract-ocr libtesseract-dev

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!pip install -q pillow pytesseract rapidfuzz

In [3]:
import sys, os, pathlib
sys.path.append(os.path.abspath('..'))  # make repo imports work

In [4]:
from app.ocr.preprocessor import preprocess_image
from app.ocr.extractor import extract_text
from rapidfuzz.distance import Levenshtein

## Helper functions for error rates

CER = Levenshtein.normalized_distance
WER is computed on a whitespace‑split token basis.

In [5]:
def cer(gt: str, pred: str) -> float:
    return Levenshtein.normalized_distance(gt, pred)

def wer(gt: str, pred: str) -> float:
    gt_words = gt.split()
    pred_words = pred.split()
    # Simple Levenshtein on word lists
    return Levenshtein.normalized_distance(' '.join(gt_words), ' '.join(pred_words))

## Load a small demo dataset

The dataset should be a directory where each image file has a matching `.txt` file containing the ground‑truth transcription. Place the folder at `data/ocr_demo/` or adjust `DATA_ROOT` accordingly.

In [6]:
import glob
DATA_ROOT = pathlib.Path('data/ocr_demo')
image_paths = sorted(glob.glob(str(DATA_ROOT / '*.png')) + glob.glob(str(DATA_ROOT / '*.jpg')) )
if not image_paths:
    print('No demo images found – add PNG/JPG files and matching .txt files to', DATA_ROOT)

No demo images found – add PNG/JPG files and matching .txt files to data\ocr_demo


## Process each image and collect metrics

In [7]:
results = []
for img_path in image_paths:
    txt_path = pathlib.Path(img_path).with_suffix('.txt')
    if not txt_path.exists():
        print(f'Ground truth missing for {img_path}')
        continue
    with open(txt_path, encoding='utf-8') as f:
        gt = f.read().strip()
    try:
        pre = preprocess_image(img_path)
        ocr_text, _ = extract_text(pre)
    except Exception as e:
        print(f'OCR failed for {img_path}: {e}')
        continue
    results.append({
        'image': img_path,
        'ground_truth': gt,
        'ocr': ocr_text.strip(),
        'cer': cer(gt, ocr_text),
        'wer': wer(gt, ocr_text)
    })

## Summary statistics

In [8]:
if results:
    avg_cer = sum(r['cer'] for r in results) / len(results)
    avg_wer = sum(r['wer'] for r in results) / len(results)
    print(f'Processed {len(results)} images')
    print(f'Average CER: {avg_cer:.2%}')
    print(f'Average WER: {avg_wer:.2%}')
else:
    print('No successful OCR results to summarise')

No successful OCR results to summarise


## Sample outputs (first three)

for r in results[:3]:
    print('Image:', r['image'])
    print('GT   :', r['ground_truth'])
    print('OCR  :', r['ocr'])
    print('CER  :', f'{r["cer"]:.2%}')
    print('WER  :', f'{r["wer"]:.2%}')
    print('-' * 40)

## Findings (to be filled after running)

* Overall OCR quality (CER/WER).
* Typical failure modes (e.g., low contrast, unusual fonts).
* Impact on downstream ingredient matching – you can feed the `ocr` column into the NLP pipeline from Notebook 1 to see how OCR errors propagate.